# Build Drivers Dimension

1. Read silver `drivers` table
1. Read gold `ref_nationality_region` table
1. Join the data from `drivers` with `ref_nationality_region` using `nationality`
1. Select the required columns
    - drivers.driver_id
    - drivers.driver_name
    - drivers.date_of_birth
    - drivers.nationality
    - ref_nationality_region.region
1. Write the transformed data to gold `dim_drivers` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

#### Step 1 - Read source tables

In [0]:
drivers_df = spark.read.table(f'{catalog_name}.{silver_schema}.drivers')
ref_region = spark.read.table(f'{catalog_name}.{gold_schema}.ref_nationality_region')


####Step 2 - Join the data from `drivers` with `ref_nationality_region` using `nationality`

In [0]:
dim_drivers = (
    drivers_df.join(
        ref_region,
        drivers_df.nationality == ref_region.nationality,
        'left'
    ).select(
        drivers_df.driver_id,
        drivers_df.driver_name,
        drivers_df.date_of_birth,
        drivers_df.nationality,
        ref_region.region
    )
)

In [0]:
# %sql
# DROP TABLE formula1.gold.dim_drivers

In [0]:
(
    dim_drivers.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
%sql
SELECT * FROM formula1.gold.dim_drivers

In [0]:
%sql
SELECT d.region, COUNT(*) as drivers_per_region FROM formula1.gold.dim_drivers d GROUP BY d.region

In [0]:
%sql
SELECT * FROM formula1.gold.dim_drivers WHERE region IS NULL